# 04 · Exploración de los datos crudos (bronze)

**Propósito:** conocer los datos tal como llegan de cada fuente antes de transformarlos: qué columnas traen, de qué
tipo son, cómo se ven los valores, cuántos faltan y qué rangos tienen. De aquí salen las reglas de limpieza de silver
y el diccionario de datos del informe.

**Entrada:** la capa bronze completa (VRA, BTS, METAR y la ficha de aeropuertos).

**Salida:** tablas descriptivas en pantalla y el diccionario de columnas de cada fuente en `docs/evidencias/datos/`.

**Requisito:** haber ejecutado los notebooks 01, 02 y 03.

In [1]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

Destino: s3://latam-bigdata-lake-728fd64c
Periodo: 2023-01 a 2025-12


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import pandas as pd

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-exploracion")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/25 07:44:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 07:44:20 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


26/11/25 07:44:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


26/11/25 07:44:24 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to logs/spark-events/app-20261125074421-0001.inprogress. This is unsupported


In [3]:
EVID = RAIZ / "docs" / "evidencias" / "datos"
EVID.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 80)


def nulos_por_columna(df, columnas):
    """Porcentaje de valores vacíos por columna (nulo, texto vacío o la palabra 'null')."""
    total = df.count()
    vacios = df.select([
        F.sum(F.when(F.col(c).isNull() | F.lower(F.trim(F.col(c))).isin("", "null"), 1).otherwise(0)).alias(c)
        for c in columnas
    ]).first().asDict()
    return total, {c: round(100 * vacios[c] / total, 2) if total else None for c in columnas}


def diccionario(df, descripciones, archivo):
    """Arma el diccionario de columnas: descripción, un ejemplo real, % vacíos y valores distintos aproximados."""
    columnas = list(descripciones)
    total, pct_nulos = nulos_por_columna(df, columnas)
    distintos = df.select([F.approx_count_distinct(c).alias(c) for c in columnas]).first().asDict()
    ejemplo = (
        df.where(F.concat_ws("", *[F.col(c) for c in columnas]).isNotNull()).limit(200).toPandas()
    )
    filas = []
    for c in columnas:
        valores = ejemplo[c].dropna()
        valores = valores[~valores.str.strip().str.lower().isin(["", "null"])]
        filas.append({
            "columna": c, "descripcion": descripciones[c],
            "ejemplo": valores.iloc[0] if len(valores) else "",
            "pct_vacios": pct_nulos[c], "valores_distintos_aprox": distintos[c],
        })
    tabla = pd.DataFrame(filas)
    tabla.to_csv(EVID / archivo, index=False)
    print(f"{total:,} filas · diccionario guardado en docs/evidencias/datos/{archivo}")
    return tabla

## 1. ANAC VRA (vuelos de Brasil)

Cada archivo mensual empieza con una línea `Atualizado em: <fecha>` y luego el encabezado entre comillas, con `;`
como separador. Todas las columnas se leen como texto: así se ve el dato exactamente como lo publica ANAC.

In [4]:
# Las 12 columnas de VRA, en el orden del archivo. Se leen como texto y se tipan después.
COLUMNAS_VRA = [
    "empresa", "numero_voo", "codigo_di", "tipo_linha", "origem", "destino",
    "partida_prevista", "partida_real", "chegada_prevista", "chegada_real", "situacao", "justificativa",
]
esquema_vra = T.StructType([T.StructField(c, T.StringType()) for c in COLUMNAS_VRA])

vra_crudo = (
    spark.read.option("sep", ";").option("quote", '"').schema(esquema_vra)
    .csv(ruta("bronze_vra"))  # year y month salen de las carpetas year=/month=
)
# Fuera la línea "Atualizado em ..." y la fila de encabezado de cada archivo
vra_crudo = en_periodo(
    vra_crudo.where(~F.col("empresa").contains("Atualizado") & (F.col("empresa") != "ICAO Empresa Aérea"))
)

In [5]:
vra_crudo.cache()
print(f"Filas de VRA en el periodo: {vra_crudo.count():,}")
vra_crudo.limit(5).toPandas()

Filas de VRA en el periodo: 2,972,195


,empresa,numero_voo,codigo_di,tipo_linha,origem,destino,partida_prevista,partida_real,chegada_prevista,chegada_real,situacao,justificativa,year,month
0,AEA,0057,0,I,LEMD,SBGR,2025-01-29 19:55:00,2025-01-29 20:45:00,2025-01-30 06:30:00,2025-01-30 06:47:00,REALIZADO,N/A,2025,1
1,AEA,0057,0,I,LEMD,SBGR,2025-01-30 19:55:00,2025-01-30 20:43:00,2025-01-31 06:30:00,2025-01-31 06:37:00,REALIZADO,N/A,2025,1
2,AEA,0057,0,I,LEMD,SBGR,2025-01-31 19:55:00,2025-01-31 20:43:00,2025-02-01 06:30:00,2025-02-01 06:30:00,REALIZADO,N/A,2025,1
3,AEA,0058,0,I,SBGR,LEMD,2025-01-01 15:05:00,2025-01-01 15:13:00,2025-01-02 01:15:00,2025-01-02 00:43:00,REALIZADO,N/A,2025,1
4,AEA,0058,0,I,SBGR,LEMD,2025-01-02 15:05:00,2025-01-02 15:16:00,2025-01-03 01:15:00,2025-01-03 00:24:00,REALIZADO,N/A,2025,1


### Diccionario de VRA

Las horas están en **hora de Brasília**, también en los vuelos internacionales. Un vuelo cancelado no tiene horas
reales: aparecen como la palabra `null`.

In [6]:
DESCRIPCION_VRA = {
    "empresa": "Código ICAO de la aerolínea (TAM = LATAM Brasil, AZU = Azul, GLO = Gol)",
    "numero_voo": "Número de vuelo",
    "codigo_di": "Código de autorización del vuelo (0 = regular)",
    "tipo_linha": "Tipo de línea: N nacional, I internacional, C carga, G otros",
    "origem": "Código ICAO del aeropuerto de origen",
    "destino": "Código ICAO del aeropuerto de destino",
    "partida_prevista": "Salida programada (hora de Brasília)",
    "partida_real": "Salida real (hora de Brasília); 'null' si no voló",
    "chegada_prevista": "Llegada programada (hora de Brasília)",
    "chegada_real": "Llegada real (hora de Brasília); 'null' si no voló",
    "situacao": "Situación del vuelo: REALIZADO o CANCELADO",
    "justificativa": "Código de justificación de la demora o cancelación (casi siempre N/A)",
}
diccionario(vra_crudo, DESCRIPCION_VRA, "diccionario_vra.csv")

2,972,195 filas · diccionario guardado en docs/evidencias/datos/diccionario_vra.csv


,columna,descripcion,ejemplo,pct_vacios,valores_distintos_aprox
0,empresa,"Código ICAO de la aerolínea (TAM = LATAM Brasil, AZU = Azul, GLO = Gol)",AEA,0.00,139
1,numero_voo,Número de vuelo,0057,0.00,11976
2,codigo_di,Código de autorización del vuelo (0 = regular),0,0.00,10
3,tipo_linha,"Tipo de línea: N nacional, I internacional, C carga, G otros",I,0.00,5
4,origem,Código ICAO del aeropuerto de origen,LEMD,0.00,527
5,destino,Código ICAO del aeropuerto de destino,SBGR,0.00,561
6,partida_prevista,Salida programada (hora de Brasília),2025-01-29 19:55:00,3.10,362915
7,partida_real,Salida real (hora de Brasília); 'null' si no voló,2025-01-29 20:45:00,4.03,1181611
8,chegada_prevista,Llegada programada (hora de Brasília),2025-01-30 06:30:00,3.10,364122
9,chegada_real,Llegada real (hora de Brasília); 'null' si no voló,2025-01-30 06:47:00,4.03,1202246


### Valores y rangos

Qué empresas vuelan más, cuántos vuelos se cancelan, qué tipos de línea hay y entre qué fechas va el periodo.

In [7]:
latam = cfg["latam"]["pasajeros"] | cfg["latam"]["carga"]
empresas = vra_crudo.groupBy("empresa").count().orderBy(F.desc("count")).limit(15).toPandas()
empresas["grupo_latam"] = empresas["empresa"].map(latam).fillna("")
empresas

,empresa,count,grupo_latam
0,AZU,912887,
1,TAM,819553,LATAM Airlines Brasil
2,GLO,702411,
3,ACN,69054,
4,PTB,37080,
5,ARG,33892,
6,TAP,28262,
7,LAN,28083,LATAM Airlines Chile
8,CMP,25100,
9,SID,19277,


In [8]:
display(vra_crudo.groupBy("situacao").count().orderBy(F.desc("count")).toPandas())
display(vra_crudo.groupBy("tipo_linha").count().orderBy(F.desc("count")).toPandas())
vra_crudo.select(
    F.min("partida_prevista").alias("primera_salida"), F.max("partida_prevista").alias("ultima_salida"),
    F.countDistinct("origem").alias("aeropuertos_origen"),
    F.max(F.length("partida_real")).alias("largo_maximo_hora_real"),
).toPandas()

,situacao,count
0,REALIZADO,2852549
1,CANCELADO,119646


,tipo_linha,count
0,N,2401163
1,I,433452
2,G,75006
3,C,62110
4,NA,464


,primera_salida,ultima_salida,aeropuertos_origen,largo_maximo_hora_real
0,2023-01-01 00:00:00,null,524,19


`largo_maximo_hora_real` mayor que 19 indica horas con fracciones de segundo (aparecen desde octubre de 2025);
en silver se recortan a `AAAA-MM-DD HH:MM:SS`.

## 2. BTS On-Time (vuelos de EE. UU.)

El archivo trae unas 110 columnas. Para el proyecto importan las de identificación del vuelo, las horas
programadas y reales y los indicadores de cancelación y desvío. Las horas vienen como texto `hhmm` en **hora local
de cada aeropuerto**; `2400` es la medianoche del día siguiente.

In [9]:
# BTS trae unas 110 columnas. El esquema se arma con los nombres del propio encabezado y todas se leen
# como texto (sin inferir tipos); después se toman y se tipan solo las que se usan.
encabezado = (
    spark.read.text(ruta("bronze_bts")).where(F.col("value").startswith('"Year"')).first()["value"]
)
nombres_bts = [n.strip('"') or f"_vacia_{i}" for i, n in enumerate(encabezado.split(","))]
# Year y Month chocan con las columnas year/month que Spark toma de las carpetas: se renombran
nombres_bts = [f"{n}_bts" if n.lower() in ("year", "month") else n for n in nombres_bts]
esquema_bts = T.StructType([T.StructField(n, T.StringType()) for n in nombres_bts])

# El encabezado de cada archivo se descarta como fila (así Spark no compara nombres con el esquema)
bts_crudo = en_periodo(
    spark.read.option("header", False).schema(esquema_bts).csv(ruta("bronze_bts"))
    .where(F.col("FlightDate") != "FlightDate")
)

In [10]:
bts_crudo.cache()
print(f"Filas de BTS en el periodo: {bts_crudo.count():,} · columnas en el archivo: {len(nombres_bts)}")
DESCRIPCION_BTS = {
    "FlightDate": "Fecha del vuelo (local del origen)",
    "Reporting_Airline": "Código IATA de la aerolínea que reporta",
    "Flight_Number_Reporting_Airline": "Número de vuelo",
    "Tail_Number": "Matrícula del avión (no se usa en el modelo)",
    "Origin": "Código IATA del aeropuerto de origen",
    "Dest": "Código IATA del aeropuerto de destino",
    "CRSDepTime": "Salida programada, hhmm en hora local del origen",
    "DepTime": "Salida real, hhmm en hora local del origen",
    "DepDelay": "Demora de salida en minutos (negativa si salió antes)",
    "CRSArrTime": "Llegada programada, hhmm en hora local del destino",
    "ArrTime": "Llegada real, hhmm en hora local del destino",
    "ArrDelay": "Demora de llegada en minutos",
    "Cancelled": "1 si el vuelo se canceló",
    "Diverted": "1 si el vuelo se desvió a otro aeropuerto",
    "CRSElapsedTime": "Duración programada en minutos",
    "Distance": "Distancia en millas",
    "CarrierDelay": "Minutos de demora atribuidos a la aerolínea (se conocen después del vuelo)",
    "WeatherDelay": "Minutos de demora atribuidos al clima (se conocen después del vuelo)",
}
diccionario(bts_crudo, DESCRIPCION_BTS, "diccionario_bts.csv")

Filas de BTS en el periodo: 20,928,579 · columnas en el archivo: 110


20,928,579 filas · diccionario guardado en docs/evidencias/datos/diccionario_bts.csv


,columna,descripcion,ejemplo,pct_vacios,valores_distintos_aprox
0,FlightDate,Fecha del vuelo (local del origen),2023-07-02,0.00,1115
1,Reporting_Airline,Código IATA de la aerolínea que reporta,9E,0.00,14
2,Flight_Number_Reporting_Airline,Número de vuelo,4900,0.00,6378
3,Tail_Number,Matrícula del avión (no se usa en el modelo),N307PQ,0.23,6787
4,Origin,Código IATA del aeropuerto de origen,DTW,0.00,359
5,Dest,Código IATA del aeropuerto de destino,DSM,0.00,359
6,CRSDepTime,"Salida programada, hhmm en hora local del origen",0946,0.00,1435
7,DepTime,"Salida real, hhmm en hora local del origen",0942,1.32,1452
8,DepDelay,Demora de salida en minutos (negativa si salió antes),-4.00,1.32,2137
9,CRSArrTime,"Llegada programada, hhmm en hora local del destino",1034,0.00,1443


In [11]:
display(bts_crudo.groupBy("Reporting_Airline").count().orderBy(F.desc("count")).limit(12).toPandas())
bts_crudo.select(
    F.min("FlightDate").alias("primera_fecha"), F.max("FlightDate").alias("ultima_fecha"),
    F.countDistinct("Origin").alias("aeropuertos_origen"),
    F.sum(F.col("Cancelled").cast("double")).alias("cancelados"),
    F.sum(F.col("Diverted").cast("double")).alias("desviados"),
    F.sum((F.col("CRSDepTime") == "2400").cast("int")).alias("salidas_programadas_2400"),
    F.sum((F.col("DepTime") == "2400").cast("int")).alias("salidas_reales_2400"),
).toPandas()

,Reporting_Airline,count
0,WN,4249769
1,DL,3020492
2,AA,2898490
3,UA,2287934
4,OO,2259642
5,YX,942776
6,MQ,806765
7,B6,746547
8,AS,736751
9,NK,719489


,primera_fecha,ultima_fecha,aeropuertos_origen,cancelados,desviados,salidas_programadas_2400,salidas_reales_2400
0,2023-01-01,2025-12-31,362,287134.0,53309.0,2,2206


## 3. METAR (clima aeronáutico)

Cada fila es un reporte de una estación: código de la estación en IEM, fecha y hora en UTC y el texto del METAR.
El código del aeropuerto (como aparece en los vuelos) viene de la carpeta `station=`.

### Cómo se lee un METAR

`SBGR 010000Z 12009KT 9999 BKN013 OVC020 19/17 Q1018` (Guarulhos):

| Parte | Significado |
|---|---|
| `SBGR` | Estación (aeropuerto de Guarulhos) |
| `010000Z` | Día 01, 00:00 UTC |
| `12009KT` | Viento desde 120° a 9 nudos |
| `9999` | Visibilidad de 10 km o más (en metros) |
| `BKN013` | Nubosidad fragmentada a 1300 pies: es el **techo** |
| `OVC020` | Cubierto a 2000 pies |
| `19/17` | Temperatura 19 °C, punto de rocío 17 °C (diferencia chica: aire casi saturado) |
| `Q1018` | Presión de 1018 hPa |

`KLAX 010053Z 20005KT 10SM FEW020 FEW150 SCT200 15/10 A3006 RMK AO2 SLP178 T01500100` (Los Ángeles):

| Parte | Significado |
|---|---|
| `20005KT` | Viento desde 200° a 5 nudos |
| `10SM` | Visibilidad de 10 millas terrestres (EE. UU. usa millas, también fracciones como `1/2SM`) |
| `FEW020 FEW150 SCT200` | Pocas nubes y dispersas: no hay techo |
| `A3006` | Presión en pulgadas de mercurio: 30,06 inHg ≈ 1018 hPa |
| `RMK ...` | Observaciones adicionales de uso interno; no se usan |

Los fenómenos aparecen como códigos: `FG` niebla, `BR` neblina, `TS` tormenta, `RA` lluvia, `DZ` llovizna, `SN`
nieve, con `-` (débil) o `+` (fuerte). `TEMPO` y `BECMG` anuncian cambios previstos.

In [12]:
esquema_metar = T.StructType([
    T.StructField("estacion_iem", T.StringType()),
    T.StructField("valid", T.StringType()),
    T.StructField("metar", T.StringType()),
])
metar_crudo = (
    spark.read.option("header", False).schema(esquema_metar).csv(ruta("bronze_metar"))
    .where(F.col("valid") != "valid")  # descarta la fila de encabezado de cada archivo
    .withColumnRenamed("station", "aeropuerto")
    .where((F.substring("valid", 1, 7) >= DESDE) & (F.substring("valid", 1, 7) <= HASTA))
)
metar_crudo.cache()
print(f"Reportes METAR en el periodo: {metar_crudo.count():,}")
metar_crudo.where(F.col("aeropuerto").isin("SBGR", "LAX")).limit(6).toPandas()

Reportes METAR en el periodo: 14,746,328


,estacion_iem,valid,metar,aeropuerto
0,LAX,2023-01-01 00:00,METAR LAX 010000Z AUTO 18006KT 15/14 RMK AO2 SLP134 60001 T01500139 IEM_GHCNH,LAX
1,LAX,2023-01-01 00:53,KLAX 010053Z 17008KT 10SM BKN014 BKN033 OVC049 15/13 A2991 RMK AO2 SLP126 T0...,LAX
2,LAX,2023-01-01 01:20,KLAX 010120Z 17009G15KT 10SM BKN008 BKN012 OVC020 15/14 A2989 RMK AO2 T01500...,LAX
3,LAX,2023-01-01 01:53,KLAX 010153Z 16006KT 10SM BKN008 OVC025 15/13 A2987 RMK AO2 SLP116 T01500133,LAX
4,LAX,2023-01-01 02:02,KLAX 010202Z 17006G16KT 10SM SCT008 BKN012 OVC024 15/13 A2987 RMK AO2 T01500133,LAX
5,LAX,2023-01-01 02:53,KLAX 010253Z 15012G18KT 10SM SCT006 BKN011 OVC018 15/14 A2983 RMK AO2 SLP101...,LAX


In [13]:
por_estacion = (
    metar_crudo.groupBy("aeropuerto")
    .agg(F.count("*").alias("reportes"), F.min("valid").alias("primero"), F.max("valid").alias("ultimo"))
    .orderBy(F.desc("reportes")).toPandas()
)
print(f"Estaciones con datos: {len(por_estacion)}")
display(por_estacion.head(10))
display(por_estacion.tail(5))

# Cuántos reportes mencionan cada fenómeno (sin parsear todavía, solo buscando el código)
metar_crudo.select(
    F.avg(F.col("metar").rlike(r"(^| )(VC|MI|BC|PR|FZ)?FG( |$)").cast("int")).alias("con_niebla"),
    F.avg(F.col("metar").rlike(r"(^| )[-+]?(VC)?TS[A-Z]*( |$)").cast("int")).alias("con_tormenta"),
    F.avg(F.col("metar").rlike(r"(^| )[-+]?(SH|TS)?RA[A-Z]*( |$)").cast("int")).alias("con_lluvia"),
    F.avg(F.col("metar").contains(" TEMPO ").cast("int")).alias("con_tempo"),
    F.avg(F.col("metar").contains("SM ").cast("int")).alias("visibilidad_en_millas"),
).toPandas()

Estaciones con datos: 453


,aeropuerto,reportes,primero,ultimo
0,PGV,92360,2023-01-01 00:15,2025-12-31 23:55
1,BJI,78227,2023-01-01 00:15,2025-12-31 23:55
2,BQK,77033,2023-01-01 00:15,2025-12-31 23:55
3,PSE,73541,2023-01-01 00:15,2025-12-31 23:55
4,KVCV,68994,2023-01-01 00:10,2025-12-31 23:55
5,KMZJ,66473,2023-01-03 16:30,2025-12-31 23:55
6,LEMD,53373,2023-01-01 00:00,2025-12-31 23:30
7,LEGE,53076,2023-01-01 00:00,2025-12-31 23:30
8,LEBL,53013,2023-01-01 00:00,2025-12-31 23:30
9,GCLP,52840,2023-01-01 00:00,2025-12-31 23:30


,aeropuerto,reportes,primero,ultimo
448,SBPB,14301,2023-01-01 10:15,2025-12-31 21:45
449,SBPF,12911,2023-01-02 10:00,2025-12-31 23:00
450,SBCX,9483,2023-01-01 10:00,2025-12-31 18:00
451,SBJE,5583,2024-04-02 13:00,2025-12-31 21:00
452,SBTC,2486,2023-01-02 13:00,2025-12-28 20:00


,con_niebla,con_tormenta,con_lluvia,con_tempo,visibilidad_en_millas
0,0.017854,0.028749,0.076852,0.005713,0.78386


In [14]:
DESCRIPCION_METAR = {
    "aeropuerto": "Código del aeropuerto como aparece en los vuelos (ICAO en Brasil, IATA en EE. UU.)",
    "estacion_iem": "Código de la estación en el archivo de IEM",
    "valid": "Fecha y hora del reporte en UTC (AAAA-MM-DD HH:MM)",
    "metar": "Texto completo del reporte METAR",
}
diccionario(metar_crudo, DESCRIPCION_METAR, "diccionario_metar.csv")

14,746,328 filas · diccionario guardado en docs/evidencias/datos/diccionario_metar.csv


,columna,descripcion,ejemplo,pct_vacios,valores_distintos_aprox
0,aeropuerto,"Código del aeropuerto como aparece en los vuelos (ICAO en Brasil, IATA en EE...",PGV,0.0,449
1,estacion_iem,Código de la estación en el archivo de IEM,PGV,0.0,437
2,valid,Fecha y hora del reporte en UTC (AAAA-MM-DD HH:MM),2023-01-01 00:15,0.0,1241295
3,metar,Texto completo del reporte METAR,KPGV 010015Z AUTO 23010KT 10SM BKN016 BKN022 OVC039 21/21 A2992 RMK AO2,0.0,14533700


## 4. Ficha de aeropuertos

La zona horaria IANA de cada aeropuerto permite pasar las horas locales de BTS a UTC respetando el horario de verano.

In [15]:
esquema_aeropuertos = T.StructType([
    T.StructField("aeropuerto", T.StringType()), T.StructField("estacion", T.StringType()),
    T.StructField("nombre", T.StringType()), T.StructField("pais", T.StringType()),
    T.StructField("latitud", T.DoubleType()), T.StructField("longitud", T.DoubleType()),
    T.StructField("zona_horaria", T.StringType()), T.StructField("tiene_metar", T.BooleanType()),
])
aeropuertos = spark.read.option("header", True).schema(esquema_aeropuertos).csv(ruta("bronze_aeropuertos"))
fichas = aeropuertos.toPandas()
print(f"Aeropuertos: {len(fichas)} · sin zona horaria: {fichas['zona_horaria'].isna().sum()}")
display(fichas.groupby(["pais", "zona_horaria"]).size().rename("aeropuertos").reset_index()
        .sort_values("aeropuertos", ascending=False).head(12))
diccionario(aeropuertos.select([F.col(c).cast("string") for c in aeropuertos.columns]), {
    "aeropuerto": "Código del aeropuerto como aparece en los vuelos",
    "estacion": "Código de la estación METAR en IEM",
    "nombre": "Nombre de la estación",
    "pais": "País (código ISO de 2 letras)",
    "latitud": "Latitud en grados",
    "longitud": "Longitud en grados",
    "zona_horaria": "Zona horaria IANA (por ejemplo America/Sao_Paulo)",
}, "diccionario_aeropuertos.csv")

Aeropuertos: 468 · sin zona horaria: 0


,pais,zona_horaria,aeropuertos
48,US,America/Chicago,120
55,US,America/New_York,105
49,US,America/Denver,42
53,US,America/Los_Angeles,38
21,BR,America/Sao_Paulo,29
50,US,America/Detroit,12
13,BR,America/Fortaleza,9
46,US,America/Anchorage,7
47,US,America/Boise,5
52,US,America/Juneau,5


468 filas · diccionario guardado en docs/evidencias/datos/diccionario_aeropuertos.csv


,columna,descripcion,ejemplo,pct_vacios,valores_distintos_aprox
0,aeropuerto,Código del aeropuerto como aparece en los vuelos,SBGR,0.0,459
1,estacion,Código de la estación METAR en IEM,SBGR,0.0,464
2,nombre,Nombre de la estación,Sao Paulo,0.0,484
3,pais,País (código ISO de 2 letras),BR,0.0,30
4,latitud,Latitud en grados,-23.43208,0.0,461
5,longitud,Longitud en grados,-46.46951,0.0,464
6,zona_horaria,Zona horaria IANA (por ejemplo America/Sao_Paulo),America/Sao_Paulo,0.0,67


## 5. Resumen de bronze

In [16]:
resumen = pd.DataFrame([
    {"fuente": "ANAC VRA", "filas": vra_crudo.count(), "entidades": vra_crudo.select("empresa").distinct().count(),
     "detalle": "aerolíneas"},
    {"fuente": "BTS On-Time", "filas": bts_crudo.count(),
     "entidades": bts_crudo.select("Reporting_Airline").distinct().count(), "detalle": "aerolíneas"},
    {"fuente": "METAR IEM", "filas": metar_crudo.count(), "entidades": len(por_estacion), "detalle": "estaciones"},
    {"fuente": "Ficha de aeropuertos", "filas": len(fichas), "entidades": fichas["pais"].nunique(),
     "detalle": "países"},
])
resumen["periodo"] = f"{DESDE} a {HASTA}"
resumen.to_csv(EVID / "resumen_bronze.csv", index=False)
resumen

,fuente,filas,entidades,detalle,periodo
0,ANAC VRA,2972195,144,aerolíneas,2023-01 a 2025-12
1,BTS On-Time,20928579,15,aerolíneas,2023-01 a 2025-12
2,METAR IEM,14746328,453,estaciones,2023-01 a 2025-12
3,Ficha de aeropuertos,468,30,países,2023-01 a 2025-12


In [17]:
spark.stop()